In [1]:
!pip install thop

In [2]:
# Adicionem aqui as bibliotecas necessárias
import pandas as pd
import os
from PIL import Image
from torchvision import datasets, transforms, models
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import json
import csv
from thop import profile, clever_format
import timm

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
path = '/content/drive/MyDrive/Projeto LIPAI/Displastia'
dataset_name = "Displastia"

In [6]:
import json
import os

def save_metrics_json(base_dir,
                     arch_name, mode, aug,
                     seed, acc, f1_macro, f1_weighted):

    dir_path = f'{base_dir}/{dataset_name}/results'
    os.makedirs(dir_path, exist_ok=True)

    results = {
        "architecture": arch_name,
        "mode": mode,
        "aug": aug,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

    aug_str = "non_aug" if not aug else "aug"

    filename = f"{arch_name}_{mode}_{aug_str}_seed{seed}.json"
    path = os.path.join(dir_path, filename)

    with open(path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Métricas salvas em: {path}")

# 1. Carregamento dos Dados

In [7]:
manifest_d = pd.read_csv(f'{path}/manifest.csv')
images_d = datasets.ImageFolder(root = f'{path}/dataset')

Divisão dos sets de acordo com o manifest

In [8]:
base_path = f'{path}/dataset'

def load_images_from_df(dataframe):
    images = []
    labels = []

    for idx, row in dataframe.iterrows():
        # Constrói o caminho completo: dataset/healthy/imagem.tif
        img_path = os.path.join(base_path, row['path'])

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            labels.append(row['label_number'])
        else:
            print(f"Erro ao carregar: {img_path}")

    return np.array(images), np.array(labels)

# 3. Criar as divisões diretamente do CSV
train_df = manifest_d[manifest_d['sets'] == 'train']
val_df   = manifest_d[manifest_d['sets'] == 'val']
test_df  = manifest_d[manifest_d['sets'] == 'test']

# 4. Carregar os dados para as variáveis
print("Carregando imagens de treino...")
x_train, y_train = load_images_from_df(train_df)

print("Carregando imagens de validação...")
x_val, y_val = load_images_from_df(val_df)

print("Carregando imagens de teste...")
x_test, y_test = load_images_from_df(test_df)

# Conferindo os formatos
print("\nDivisão concluída:")
print(f"Treino: {x_train.shape}, {y_train.shape}")
print(f"Val:    {x_val.shape}, {y_val.shape}")
print(f"Teste:  {x_test.shape}, {y_test.shape}")

Carregando imagens de treino...
Carregando imagens de validação...
Carregando imagens de teste...

Divisão concluída:
Treino: (144, 250, 450, 3), (144,)
Val:    (38, 250, 450, 3), (38,)
Teste:  (46, 250, 450, 3), (46,)


## Transformações e Data Augmentation

Não queremos distorções brucas nas imagens histológicas, portanto, escolhemos aplicar apenas 3 operações geometricas.

* RandomHorizontalFlip: Espelhamento horizontal.
* RandomVerticalFlip: Espelhamento vertical.
* RandomRotation: Rotação suave de 15 graus.

In [9]:
transform_basic = transforms.Compose([ # Básica, sem augmentation
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

transform_augmentation = transforms.Compose([ # Com augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

## DataLoaders

Estou ultilizando uma classe auxiliar HistologyDataset, que envolve os arrays numpy, e os alinha com o pipeline do torchvision.transforms.


In [10]:
class HistologyDataset(Dataset):
    def __init__(self, images_array, labels_array, transform=None):
        self.images = images_array
        self.labels = labels_array
        self.transform = transform

    def __len__(self):
        # Tamanho total do dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # Pega a imagem em formato Numpy
        img_np = self.images[idx]
        label = self.labels[idx]

        # Converte BGR para RGB
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

        # Converte o array para um objeto Imagem (PIL)
        img_pil = Image.fromarray(img_rgb)

        # Aplica as transformações
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
        return img_tensor, label

In [11]:
BATCH_SIZE = 32

# Aplicação das transformações no treino.
train_dataset = HistologyDataset(x_train, y_train, transform=transform_augmentation)
val_dataset = HistologyDataset(x_val, y_val, transform=transform_basic)
test_dataset = HistologyDataset(x_test, y_test, transform=transform_basic)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders construídos!")
print(f"Total de lotes no treino: {len(train_loader)}")

DataLoaders construídos!
Total de lotes no treino: 5


# 2. Aplicação dos Algoritmos

Estou definindo uma função que permite aplicar os algoritmos de forma dinamica. Conforme as orientações, vamos usar:

* FS - From Scratch: Possui pesos aleatórios e aprende as caracteristicas visuais do zero.
* PT-ALL - Fine-tuning Completo: Possui pesos pré-treinados da ImageNet e será executado com todas as camadas treináveis.
* PT-FC - Backbone Congelado: Possui pesos pré-treinados da ImageNet e será executado com apenas o classificador treinável.

In [12]:
def build_model(name, mode, num_classes):
    """Fábrica de modelos. """

    if mode == 'FS':
        model = timm.create_model(name, pretrained=False, num_classes=num_classes)
    elif mode == 'PT-ALL':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
    elif mode == 'PT-FC':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
        # Congela os parâmetros
        for param in model.parameters():
            param.requires_grad = False
        # Descongela os parâmetros da última camada
        for param in model.get_classifier().parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Modo de treinamento {mode} não encontrado.")
    return model

# Como displasia é uma classificação binária, num_classes=2
modelo_teste = build_model('convnextv2_atto', 'PT-FC', num_classes=2)

total_params = sum(p.numel() for p in modelo_teste.parameters())
trainable_params = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)

print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis (PT-FC): {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Total de parâmetros: 3,388,042
Parâmetros treináveis (PT-FC): 642


## 3. Treinamento

Da mesma forma que foi feito na célula anterior, estou criando uma função que contém a logica do laço de treinamento e avaliação. Dessa forma, a execução dos 72 experimentos vai ficar muito dinamica.

In [13]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, num_epochs=50, device='cuda'):
    """ Função de treinamento e validação que retorna o melhor modelo e as métricas."""

    model.to(device)
    best_val_acc = 0.0
    best_model_state = None
    best_metrics = {}
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

        # Fase de Treinamento
        model.train()
        running_loss = 0.0
        all_train_preds, all_train_labels = [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad() # Zera o erro anterior
            outputs = model(inputs) # Previsão
            loss = criterion(outputs, labels) # Calcula o erro
            loss.backward() # Calcula os gradientes
            optimizer.step() # Ajusta os pesos

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

        # Fase de Validação
        model.eval()
        val_loss = 0.0
        all_val_preds, all_val_labels = [], []

        # Desliga o cálculo dos gradientes
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = accuracy_score(all_val_labels, all_val_preds)

        # Salva o histórico
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        # Atualiza melhor modelo
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = model.state_dict().copy()  # Salva cópia dos pesos
            best_metrics = {
                'epoch': epoch + 1,
                'val_acc': epoch_val_acc,
                'val_f1_macro': f1_score(all_val_labels, all_val_preds, average='macro'),
                'val_f1_weighted': f1_score(all_val_labels, all_val_preds, average='weighted')
            }

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    return best_model_state, best_metrics, history

## GFLOPs e Parâmetros da Arquitetura

Antes de rodar os experimentos, calculo os GFLOPs e o número de parâmetros de cada arquitetura, valores fixos para entrada 224×224 que vão para os gráficos comparativos.

In [14]:
def get_model_complexity(model_name, input_size=(1,3,224,224), device='cuda'):
    """Retorna GFLOPs, parâmetros para uma arquitetura."""

    # Cria modelo com pesos aleatórios
    model = timm.create_model(model_name, pretrained=False, num_classes=2)
    model.eval()
    model = model.to(device)
    input_tensor = torch.randn(*input_size).to(device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops_gflops = flops / 1e9
    params_m = params / 1e6
    return flops_gflops, params_m

# Armazena a complexidade de cada arquitetura
complexidade = {}
arquiteturas = ['convnextv2_atto', 'convnextv2_pico']

print("Calculando GFLOPs e parâmetros para cada arquitetura...")

for arch in arquiteturas:
    gflops, params_m = get_model_complexity(arch)
    complexidade[arch] = {'gflops': gflops, 'params_m': params_m}
    print(f"{arch}: {params_m:.2f}M parâmetros, {gflops:.2f} GFLOPs")

Calculando GFLOPs e parâmetros para cada arquitetura...
convnextv2_atto: 3.37M parâmetros, 0.55 GFLOPs
convnextv2_pico: 8.52M parâmetros, 1.37 GFLOPs


## Execução do Loop de Treinamento

Chama as funções definidas acima para executar os experimentos, mudando os modos, condições de augmentation e seeds.

Além de salvar as métricas e historicos, consolido tudo em um arquivo CSV.

In [ ]:
import os
import csv
import json

arquiteturas = ['convnextv2_atto', 'convnextv2_pico']
modos = ['FS', 'PT-FC', 'PT-ALL']
augmentations_list = [True, False]
seeds = [42, 123, 2025]

num_classes_dataset = len(np.unique(y_train))
criterion = nn.CrossEntropyLoss()

# Caminho para salvar os resultados
result_dir = f'{path}/{dataset_name}/results'
os.makedirs(result_dir, exist_ok=True)

# Arquivo CSV
csv_path = f'{result_dir}/resultados_consolidados.csv'
file_exists = os.path.isfile(csv_path)
with open(csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists or os.stat(csv_path).st_size == 0:
        writer.writerow([
            'seed', 'dataset', 'model', 'training_mode', 'augmentation',
            'acc_test', 'f1_macro_test', 'f1_weighted_test',
            'num_params', 'gflops', 'best_epoch', 'val_acc_best'
        ])

print("Verificando experimentos concluídos e retomando...\n")

for arch in arquiteturas:
    # Complexidade calculada
    gflops = complexidade[arch]['gflops']
    params_m = complexidade[arch]['params_m']

    for modo in modos:
        for aug in augmentations_list:

            # Define transformação de treino conforme augmentation
            transformacao_atual = transform_augmentation if aug else transform_basic
            train_dataset_dinamico = HistologyDataset(x_train, y_train, transform=transformacao_atual)
            train_loader_dinamico = DataLoader(train_dataset_dinamico, batch_size=BATCH_SIZE, shuffle=True)

            for seed in seeds:

                # sistema de retomada do aprendizado
                aug_str = "aug" if aug else "non_aug"
                hist_filename = f"{arch}_{modo}_{aug_str}_seed{seed}_history.json"
                hist_path = os.path.join(result_dir, hist_filename)

                # Se o arquivo já existe no Drive, o modelo ja foi salvo
                if os.path.exists(hist_path):
                    print(f"Pulando: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed} (Já concluído)")
                    continue

                print(f"\n{'='*30}")
                print(f"Experimento: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed}")
                print(f"{'='*30}")

                # Fixa seeds
                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                np.random.seed(seed)
                random.seed(seed)
                torch.backends.cudnn.deterministic = True

                # Constrói modelo
                modelo_atual = build_model(name=arch, mode=modo, num_classes=num_classes_dataset)

                # Otimizador
                optimizer = optim.Adam(modelo_atual.parameters(), lr=1e-4)

                # Treinamento
                best_state, best_metrics, history = train_and_evaluate(
                    model=modelo_atual,
                    train_loader=train_loader_dinamico,
                    val_loader=val_loader,
                    criterion=criterion,
                    optimizer=optimizer,
                    num_epochs=50,
                    device='cuda' if torch.cuda.is_available() else 'cpu'
                )

                # Carrega o melhor modelo
                modelo_atual.load_state_dict(best_state)

                # Avaliação
                modelo_atual.eval()
                all_test_preds = []
                all_test_labels = []

                with torch.no_grad():
                    for inputs, labels in test_loader:
                        inputs, labels = inputs.to(device), labels.to(device)
                        outputs = modelo_atual(inputs)
                        _, preds = torch.max(outputs, 1)
                        all_test_preds.extend(preds.cpu().numpy())
                        all_test_labels.extend(labels.cpu().numpy())

                acc_test = accuracy_score(all_test_labels, all_test_preds)
                f1_macro_test = f1_score(all_test_labels, all_test_preds, average='macro')
                f1_weighted_test = f1_score(all_test_labels, all_test_preds, average='weighted')

                # Salva curvas de aprendizado
                with open(hist_path, 'w') as f:
                    json.dump(history, f, indent=4)

                # Salva predições do teste
                pred_filename = f"{arch}_{modo}_{aug_str}_seed{seed}_predictions.json"
                pred_path = os.path.join(result_dir, pred_filename)
                with open(pred_path, 'w') as f:
                    json.dump({
                        # Converte em int pra poder salvar no json
                        'y_true': [int(v) for v in all_test_labels],
                        'y_pred': [int(v) for v in all_test_preds]
                    }, f, indent=4)

                # Escreve CSV
                with open(csv_path, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        seed,
                        dataset_name,
                        arch,
                        modo,
                        'sim' if aug else 'nao',
                        acc_test,
                        f1_macro_test,
                        f1_weighted_test,
                        params_m,
                        gflops,
                        best_metrics['epoch'],
                        best_metrics['val_acc']
                    ])

                # Salva as métricas de teste no formato JSON
                save_metrics_json(
                    base_dir=path,
                    arch_name=arch,
                    mode=modo,
                    aug=aug,
                    seed=seed,
                    acc=acc_test,
                    f1_macro=f1_macro_test,
                    f1_weighted=f1_weighted_test
                )

print(f"\nExperimentos concluídos!")
print(f"CSV salvo/atualizado em: {csv_path}")
print(f"Históricos e predições salvos na pasta: {result_dir}")

Verificando experimentos concluídos e retomando...

Pulando: Arq=convnextv2_atto | Modo=FS | Aug=True | Seed=42 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=FS | Aug=True | Seed=123 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=FS | Aug=True | Seed=2025 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=FS | Aug=False | Seed=42 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=FS | Aug=False | Seed=123 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=FS | Aug=False | Seed=2025 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=True | Seed=42 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=True | Seed=123 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=True | Seed=2025 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=False | Seed=42 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=False | Seed=123 (Já concluído)
Pulando: Arq=convnextv2_atto | Modo=PT-FC | Aug=False | Seed=2025 (Já concluído)
Pulando: Arq

# 4. Comparativos Globais



In [16]:
# Para cada dataset, o grupo deverá apresentar comparações globais usando: <br>
#  * Gráfico de barras do F1-score por arquitetura, modo de treinamento e uso de augmentation. <br>
#  * Tabela com média e desvio padrão das repetições. <br>
#  * Gráfico com número de parâmetros por arquitetura. <br>
#  * Gráfico com complexidade computacional em GFLOPs considerando entrada 224 × 224.